# Student Skill & Project Recommendation System

This notebook implements a complete end‑to‑end pipeline for recommending projects to students based on their academic year, department, current skills, interests and available hours per week. It follows the specifications provided, using only this notebook for all code.

In [2]:
# Install required packages (run once)
%pip install -q catboost openpyxl joblib matplotlib seaborn scikit-learn pandas

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os, re, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from catboost import CatBoostRegressor
import joblib

## 1. Load and Inspect the Dataset

In [4]:
DATA_FILE = 'student_recommendation_clean_dataset.xlsx'
df = pd.read_excel(DATA_FILE)
print('Dataset shape:', df.shape)
print('Columns:', df.columns.tolist())

Dataset shape: (10000, 10)
Columns: ['Student_ID', 'Year', 'Department', 'Current_Skills', 'Interest', 'Available_Hours_Per_Week', 'Recommended_Skills', 'Recommended_Project', 'Recommendation_Reason', 'Recommended_Tools']


## 2. Basic Data Cleaning

In [6]:
# Missing values
print("Missing values per column:", df.isnull().sum())
# Duplicates
print('Duplicate rows:', df.duplicated().sum())

Missing values per column: Student_ID                  0
Year                        0
Department                  0
Current_Skills              0
Interest                    0
Available_Hours_Per_Week    0
Recommended_Skills          0
Recommended_Project         0
Recommendation_Reason       0
Recommended_Tools           0
dtype: int64
Duplicate rows: 0


## 3. Generate Project Catalog Automatically

In [8]:
CATALOG_FILE = "project_catalog.xlsx"

def parse_skills(skill_str):
    if pd.isna(skill_str):
        return []
    parts = re.split(r'[;,|]\s*', str(skill_str))
    return [p.strip().lower() for p in parts if p.strip()]

def most_common(lst):
    if not lst:
        return ""
    return pd.Series(lst).mode().iloc[0]

catalog_rows = []

for project, g in df.groupby("Recommended_Project"):
    all_skills = []

    for skills in g["Current_Skills"].dropna():
        all_skills.extend(parse_skills(skills))

    common_skills = (
        pd.Series(all_skills).value_counts().head(5).index.tolist()
        if all_skills else []
    )

    years = sorted(g["Year"].dropna().astype(str).unique())

    hours = pd.to_numeric(
        g["Available_Hours_Per_Week"],
        errors="coerce"
    ).median()

    catalog_rows.append({
        "Project_Name": project,
        "Interest": most_common(g["Interest"].dropna().tolist()),
        "Required_Skills": ", ".join(common_skills),
        "Suitable_Years": ", ".join(years),
        "Estimated_Hours_Per_Week": round(hours) if pd.notna(hours) else None,
        "Recommended_Tools": most_common(
            g["Recommended_Tools"].dropna().tolist()
        )
    })

catalog = pd.DataFrame(catalog_rows)

catalog.to_excel(CATALOG_FILE, index=False)

print("Project catalog created with", len(catalog), "projects")
print(catalog.head())

Project catalog created with 49 projects
                        Project_Name          Interest  \
0      3D Component Design Portfolio        CAD Design   
1             Arduino Sensor Monitor  Embedded Systems   
2  Biological Data Analysis Notebook     Biotechnology   
3          Bioprocess Data Dashboard     Biotechnology   
4    Biotechnology Literature Review     Biotechnology   

                                  Required_Skills  \
0  engineering drawing basics, project experience   
1                    c basics, project experience   
2      python basics, biology, project experience   
3              biology, excel, project experience   
4              biology basics, project experience   

                           Suitable_Years  Estimated_Hours_Per_Week  \
0  1st Year, 2nd Year, 3rd Year, 4th Year                         6   
1  1st Year, 2nd Year, 3rd Year, 4th Year                         6   
2            2nd Year, 3rd Year, 4th Year                        12   
3      

## 4. Suitability Scoring (internal heuristic)

In [9]:
WEIGHTS = {
    'interest': 0.35,
    'skills': 0.30,
    'year': 0.20,
    'time': 0.15
}

def clean_text(val):
    return str(val).strip().lower() if pd.notna(val) else ''

def parse_skills_set(val):
    return set(parse_skills(val))

def parse_year(val):
    m = re.search(r'\d+', str(val))
    return int(m.group()) if m else None

def calculate_scores(student, project):
    interest_score = 100.0 if clean_text(student['Interest']) == clean_text(project['Interest']) else 0.0
    student_skills = parse_skills_set(student['Current_Skills'])
    required_skills = parse_skills_set(project['Required_Skills'])
    if required_skills:
        skills_score = len(student_skills & required_skills) / len(required_skills) * 100
    else:
        skills_score = 0.0
    student_year = parse_year(student['Year'])
    suitable_years = {int(y) for y in re.findall(r'\d+', str(project['Suitable_Years']))}
    year_score = 100.0 if student_year in suitable_years else 0.0
    available = pd.to_numeric(student['Available_Hours_Per_Week'], errors='coerce')
    required = pd.to_numeric(project['Estimated_Hours_Per_Week'], errors='coerce')
    time_score = min(available / required, 1.0) * 100 if pd.notna(available) and pd.notna(required) and required > 0 else 0.0
    total = (WEIGHTS['interest'] * interest_score +
             WEIGHTS['skills'] * skills_score +
             WEIGHTS['year'] * year_score +
             WEIGHTS['time'] * time_score)
    return {
        'Interest_Score': round(interest_score, 2),
        'Skills_Score': round(skills_score, 2),
        'Year_Score': round(year_score, 2),
        'Time_Score': round(time_score, 2),
        'Suitability_Score': round(total, 2)
    }

catalog = pd.read_excel(CATALOG_FILE)
records = []
for _, student in df.iterrows():
    for _, proj in catalog.iterrows():
        scores = calculate_scores(student, proj)
        rec = {
            'Student_ID': student['Student_ID'],
            'Year': student['Year'],
            'Department': student['Department'],
            'Current_Skills': student['Current_Skills'],
            'Interest': student['Interest'],
            'Available_Hours_Per_Week': student['Available_Hours_Per_Week'],
            'Historical_Recommended_Project': student['Recommended_Project'],
            'Candidate_Project': proj['Project_Name'],
            **scores
        }
        records.append(rec)
scored_df = pd.DataFrame(records)
OUTPUT_FILE = 'student_project_scored.xlsx'
scored_df.to_excel(OUTPUT_FILE, index=False)
print('Scoring completed – saved to', OUTPUT_FILE)

Scoring completed – saved to student_project_scored.xlsx


## 5. Feature Engineering & Train‑Test Split

In [10]:
scored = pd.read_excel(OUTPUT_FILE)
X = scored[[
    'Year', 'Department', 'Current_Skills', 'Interest', 'Available_Hours_Per_Week'
]]
y = scored['Suitability_Score']
categorical_cols = ['Year', 'Department', 'Current_Skills', 'Interest']
preprocessor = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)], remainder='passthrough')
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print('Train size:', X_train.shape[0], 'Test size:', X_test.shape[0])

Train size: 392000 Test size: 98000


## 6. Train Regression Models

In [12]:
models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, random_state=42),
    'CatBoost': CatBoostRegressor(iterations=300, depth=6, learning_rate=0.05, loss_function='RMSE', verbose=0, random_state=42),
    'KNN': KNeighborsRegressor(n_neighbors=5)
}
results = {}
for name, mdl in models.items():
    pipe = Pipeline([('preprocess', preprocessor), ('model', mdl)])
    pipe.fit(X_train, y_train)
    preds = pipe.predict(X_test)
    mae = mean_absolute_error(y_test, preds)
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    results[name] = {'pipeline': pipe, 'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"=== {name} ===")
    print("MAE : {mae:.3f}")
    print(f"RMSE: {rmse:.3f}")
    print("fR2  : {r2:.3f}")

=== Random Forest ===
MAE : {mae:.3f}
RMSE: 12.450
fR2  : {r2:.3f}
=== CatBoost ===
MAE : {mae:.3f}
RMSE: 12.410
fR2  : {r2:.3f}


KeyboardInterrupt: 

## 7. Model Comparison & Selection

In [ ]:
best_name = max(results, key=lambda k: results[k]['R2'])
best_model = results[best_name]['pipeline']
print('Best model selected:', best_name)

## 8. Persist the Best Model

In [ ]:
MODEL_PKL = 'best_student_recommendation_model.pkl'
joblib.dump(best_model, MODEL_PKL)
print('Model saved to', MODEL_PKL)

## 9. Verify Model Reload and Predict for a New Student

In [ ]:
loaded_model = joblib.load(MODEL_PKL)
new_student = pd.DataFrame([{
    'Year': '3rd Year',
    'Department': 'IT',
    'Current_Skills': 'Python, SQL',
    'Interest': 'Data Science',
    'Available_Hours_Per_Week': 10
}])
pred_score = loaded_model.predict(new_student)[0]
print('Predicted suitability score for example student (internal):', round(pred_score,2))

## 10. Recommendation Function
The function `recommend_projects` returns the top‑N project recommendations for a given student profile. It hides the internal suitability scores from the end user.

In [ ]:
def recommend_projects(year, department, current_skills, interest, available_hours, top_n=5):
    """Return a list of recommendations (project name, skills, tools, explanation)."""
    catalog = pd.read_excel(CATALOG_FILE)
    student_df = pd.DataFrame([{
        'Year': year,
        'Department': department,
        'Current_Skills': current_skills,
        'Interest': interest,
        'Available_Hours_Per_Week': available_hours
    }])
    repeated = pd.concat([student_df]*len(catalog), ignore_index=True)
    repeated['Project_Name'] = catalog['Project_Name'].values
    pred_scores = best_model.predict(repeated.drop(columns=['Project_Name']))
    catalog['Predicted_Score'] = pred_scores
    top = catalog.nlargest(top_n, 'Predicted_Score')
    recommendations = []
    for _, row in top.iterrows():
        explanation = f"Interest match, skill overlap, year compatibility and time availability gave a score of {row['Predicted_Score']:.1f}."
        recommendations.append({
            'Project_Name': row['Project_Name'],
            'Recommended_Skills': row['Required_Skills'],
            'Recommended_Tools': row['Recommended_Tools'],
            'Explanation': explanation
        })
    return recommendations

# Test the function with a few profiles
samples = [
    ('2nd Year', 'CSE', 'Python, HTML', 'Web Development', 8),
    ('1st Year', 'EEE', 'C, Electronics', 'Electronics', 12),
    ('3rd Year', 'IT', 'SQL, Data Analysis', 'Data Science', 15)
]
for s in samples:
    recs = recommend_projects(*s)
    print('\nStudent profile:', s)
    for r in recs:
        print('- Project:', r['Project_Name'])
        print('  Skills to learn:', r['Recommended_Skills'])
        print('  Tools:', r['Recommended_Tools'])
        print('  Why:', r['Explanation'])

## 11. Summary of Generated Files
- `student_recommendation_full.ipynb` – this notebook
- `student_recommendation_clean_dataset.xlsx` – original data (unchanged)
- `project_catalog.xlsx` – derived project catalog
- `student_project_scored.xlsx` – all student‑project combos with suitability scores
- `best_student_recommendation_model.pkl` – persisted pipeline with best regressor